# Matrices · Locate a point with a QR solve
**CAS Computational Form-finding · Block Research Group · ETH Zürich**

**Why this is needed:** geometric and structural constraints often become several
linear equations for shared unknowns. A matrix stores their coefficients; a
factorisation helps solve them and exposes when the equations are independent.
Before a truss, solve **two equations for a point's x and y coordinates**.

Here `x + y = 3` and `x − y = 1`, with coordinates and right-hand sides in metres.
**Predict:** where do the two lines intersect?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from compas.geometry import Line, Pointcloud
from compas_notebook import Viewer
from compas_notebook.config import Config

A = np.array([[1.0, 1.0], [1.0, -1.0]])
b = np.array([3.0, 1.0])  # m
print("A =\n", A, "\nb =", b)
assert np.linalg.matrix_rank(A) == 2, "The two constraints must be independent."


## Factor the matrix, then solve
QR decomposes `A = Q @ R`. Q is orthogonal (`Q.T @ Q = I`) and R is upper
triangular. Multiplying `A @ xy = b` by `Q.T` gives `R @ xy = Q.T @ b`.
`solve` finds the coordinates without explicitly forming a matrix inverse.
NumPy is the numerical solver; COMPAS constructs the displayed geometry.


In [ ]:
Q, R = np.linalg.qr(A)
xy = np.linalg.solve(R, Q.T @ b)
residual = A @ xy - b
assert np.allclose(Q.T @ Q, np.eye(2))
assert np.allclose(Q @ R, A)
assert np.allclose(R, np.triu(R))
assert np.max(np.abs(residual)) < 1e-12
assert np.allclose(xy, [(b[0] + b[1]) / 2, (b[0] - b[1]) / 2])
print("Intersection [m]:", xy)
print("Equation residual [m]:", residual)


## Display the constraints and the computed point
The first line is purple, the second orange. The blue point is the numerical
solution. Construct line endpoints from b, so changing the right-hand sides
updates the picture as well as the calculation.


In [ ]:
line_sum = Line([0, b[0], 0], [b[0], 0, 0])
line_difference = Line([0, -b[1], 0], [3, 3 - b[1], 0])
intersection = Pointcloud([[float(xy[0]), float(xy[1]), 0]])

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [4, -5, 6]
config.view.camera.target = [1.5, 1, 0]
view = Viewer(config=config)
view.scene.add(line_sum, color="#5552bb")
view.scene.add(line_difference, color="#e89b43")
view.scene.add(intersection, color="#348f9e", pointsize=0.15)
view.show()


## Predict → change → check
Starting solution: **(x, y) = (2, 1) m**.

1. Change the first right-hand side from 3 to 4 m. Predict which line moves and where the new intersection lies.
2. Write `x − y = 0` as a matrix row and predict the point before solving.
3. What happens if both rows of A are identical? Check its rank before trying to solve.
4. Explain how a joint-equilibrium equation could replace a geometric line equation later.

The line display is designed for this A. If you change its coefficients, update
the plotted endpoints to express your new line equations.


In [ ]:
# Safe experiment: a new right-hand side with the same factorisation.
trial_b = np.array([3.0, 0.0])
trial_xy = np.linalg.solve(R, Q.T @ trial_b)
assert np.allclose(A @ trial_xy, trial_b)
assert np.allclose(trial_xy, [1.5, 1.5])
print("Trial intersection [m]:", trial_xy)


## Sources
Original course geometric example. Mathematical APIs:
[NumPy QR](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html),
[solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html), and
[matrix rank](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html).
Geometry: [COMPAS](https://compas.dev/compas/2.15.1/api/compas.geometry.html); viewer pattern: [COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
